# Region-reactive: compute only what's on screen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/10_region_reactive.ipynb)

The view syncs its visible region back to Python, so you can **observe `location` and recompute as the user pans** — the loop a static browser can't close. Here a per-base score (stand-in for coverage from a BAM, a motif scan, GC content, …) is computed only over the window in view, at a bin size that adapts to the zoom level. Nothing is precomputed genome-wide; the kernel answers for exactly what's asked.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The (expensive) per-base score

`score` stands in for something you would not want to run across a whole genome up front — read depth from a `pysam` pileup, a PWM scan, GC content. `compute_windows` evaluates it only between `start` and `end` and bins to ~200 points across the view, so zooming in *raises* the resolution instead of just cropping a fixed-resolution file.

In [ ]:
import numpy as np
import pandas as pd

# two peaks in the score landscape, to have something to find
PEAKS = [(29_845_000, 2500, 1.0), (29_905_000, 6000, 0.7)]


def score(pos):
    pos = np.asarray(pos, dtype=float)
    y = 0.12 * np.sin(pos / 700.0)  # background
    for center, width, height in PEAKS:
        y = y + height * np.exp(-0.5 * ((pos - center) / width) ** 2)
    return y


def compute_windows(chrom, start, end):
    span = max(end - start, 1)
    binsize = max(50, span // 200)  # ~200 bins across the view
    edges = np.arange(start, end, binsize)
    rows = [
        {
            "chrom": chrom,
            "start": int(edge),
            "end": int(min(edge + binsize, end)),
            "score": round(float(score(np.arange(edge, min(edge + binsize, end))).mean()), 3),
        }
        for edge in edges
    ]
    return pd.DataFrame(rows)


compute_windows("10", 29_838_000, 29_858_000).head()

## Recompute on every pan

`on_location` parses the view's current locstring and re-renders the computed track for that window. `view.observe(..., "location")` fires it whenever the region changes — dragging in the UI, or setting `view.location` from code. `parse_loc` returns `None` for a gene-name or empty location, so those are simply skipped.

In [ ]:
import re

from jbrowse_anywidget import LinearGenomeView, make_assembly

grch38 = make_assembly(
    "GRCh38",
    "https://s3.amazonaws.com/jbrowse.org/genomes/GRCh38/fasta/GRCh38.fa.gz",
    aliases=["hg38"],
)


def parse_loc(loc):
    m = re.match(r"^\s*([^:\s]+)\s*:\s*([\d,]+)\s*\.\.\s*([\d,]+)", loc or "")
    if not m:
        return None
    return m.group(1), int(m.group(2).replace(",", "")), int(m.group(3).replace(",", ""))


def render_region(chrom, start, end):
    view.tracks = []  # replace with the freshly computed window
    view.add_features(
        compute_windows(chrom, start, end),
        name="score (visible region)",
        track_id="signal",
        color="jexl:get(feature,'score') > 0.5 ? '#c62828' : get(feature,'score') > 0.25 ? '#f9a825' : '#90a4ae'",
    )


def on_location(change):
    region = parse_loc(change["new"])
    if region:
        render_region(*region)


view = LinearGenomeView(assembly=grch38, location="10:29,838,000..29,858,000")
view.observe(on_location, "location")
render_region(*parse_loc(view.location))  # initial fill
view

Driving `location` from code fires the observer, so the track recomputes for the new window. Zooming out widens the bins; zooming in sharpens them — the resolution follows the view:

In [ ]:
view.location = "10:29,890,000..29,920,000"  # zoom to the second peak
len(view.tracks[0]["adapter"]["features"]), "bins computed for this window"